## 📊 Visualización Interactiva de Anomalías CCL (Versión Local)

Este notebook permite explorar de forma local e interactiva los resultados de detección de anomalías en la señal CCL, sin necesidad de Streamlit.

### Funciones:
- Visualización con `Plotly`
- Controles interactivos con `ipywidgets`
- Ajuste de umbral
- Identificación visual de anomalías
- Exportación a CSV


### 💻 Celda 2 - Carga de datos y librerías

In [1]:
import pandas as pd
import plotly.graph_objs as go
import ipywidgets as widgets
from IPython.display import display
from io import StringIO
import base64

# Carga de datos
df = pd.read_csv(r"C:\Developer\fundamentos\data\ccl_anomaly_scores.csv")

# Preparar listas de pozos y etapas
pozos = sorted(df["pozo"].dropna().unique())

def get_etapas(pozo):
    return sorted(df[df["pozo"] == pozo]["etapa"].dropna().unique())


### 🔧 Celda 3 - Controles interactivos

In [ ]:
# Widgets de selección
pozo_sel = widgets.Dropdown(options=pozos, description="Pozo:")
etapa_sel = widgets.Dropdown(description="Etapa:")
modelo_sel = widgets.Dropdown(
    options=["score_iso", "score_knn", "anomaly_lof", "anomaly_svm"],
    description="Modelo:"
)


# Actualizar etapas según pozo
def on_pozo_change(change):
    etapa_sel.options = get_etapas(change['new'])
pozo_sel.observe(on_pozo_change, names='value')
on_pozo_change({'new': pozo_sel.value})

# Umbral dinámico (se actualizará luego)
umbral_slider = widgets.FloatSlider(description='Umbral', step=0.01, layout=widgets.Layout(width='50%'))

# Mostrar widgets
display(pozo_sel, etapa_sel, modelo_sel, umbral_slider)


Dropdown(description='Pozo:', options=('BPO-2703',), value='BPO-2703')

Dropdown(description='Etapa:', options=('E1', 'E10', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', '…

Dropdown(description='Modelo:', options=('score_iso', 'score_knn'), value='score_iso')

FloatSlider(value=0.0, description='Umbral', layout=Layout(width='50%'), step=0.01)

### 🧠 Celda 4 - Función de visualización

In [3]:
def graficar_anomalias(pozo, etapa, modelo, umbral):
    df_sel = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values("DEPT").copy()
    df_sel["is_anomaly"] = df_sel[modelo] > umbral

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_sel["CCL_norm"],
        y=df_sel["DEPT"],
        mode='lines',
        name='CCL Normalizado',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        x=df_sel[modelo],
        y=df_sel["DEPT"],
        mode='lines',
        name=f'Score {modelo.replace("score_", "").upper()}',
        line=dict(color='orange', dash='dot')
    ))

    fig.add_trace(go.Scatter(
        x=df_sel[df_sel["is_anomaly"]][modelo],
        y=df_sel[df_sel["is_anomaly"]]["DEPT"],
        mode='markers',
        name='⚠️ Anomalías',
        marker=dict(size=8, color='red', symbol='x'),
    ))

    fig.update_layout(
        title=f"Pozo: {pozo} | Etapa: {etapa} | Modelo: {modelo.replace('score_', '').upper()}",
        yaxis_title="Profundidad (DEPT)",
        xaxis_title="Valor",
        yaxis_autorange='reversed',
        height=600,
        margin=dict(l=20, r=20, t=40, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    fig.show()
    return df_sel


### ⚙️ Celda 5 - Botón de actualización y exportación

In [4]:
out = widgets.Output()
display(out)

def actualizar_grafico(*args):
    pozo = pozo_sel.value
    etapa = etapa_sel.value
    modelo = modelo_sel.value

    df_sel = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values("DEPT")
    max_score = df_sel[modelo].max()
    min_score = df_sel[modelo].min()
    default = min_score + (max_score - min_score) * 0.85
    umbral_slider.min = float(min_score)
    umbral_slider.max = float(max_score)
    umbral_slider.value = float(default)

    with out:
        out.clear_output()
        df_plot = graficar_anomalias(pozo, etapa, modelo, umbral_slider.value)
        mostrar_exportacion(df_plot)

def on_umbral_change(change):
    actualizar_grafico()

def mostrar_exportacion(df_plot):
    csv = df_plot.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    href = f'<a download="export_anomalias.csv" href="data:text/csv;base64,{b64}">📥 Descargar CSV</a>'
    display(widgets.HTML(value=href))

# Eventos
pozo_sel.observe(actualizar_grafico, names='value')
etapa_sel.observe(actualizar_grafico, names='value')
modelo_sel.observe(actualizar_grafico, names='value')
umbral_slider.observe(on_umbral_change, names='value')

# Primera visualización
actualizar_grafico()


Output()